# DeepSeek-R1 reinforcement learning: GRPO from first principles

This notebook implements the core Group Relative Policy Optimization (GRPO) loop used in DeepSeek-R1 on a tiny autoregressive policy. It uses CPU PyTorch and no RL training framework.

**Anchor paper:** DeepSeek-AI et al., [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948), arXiv:2501.12948v2. Related version of record: [Nature 645, 633-638 (2025)](https://www.nature.com/articles/s41586-025-09422-z). The local PDF is `papers/deepseek-r1-arxiv-2501.12948v2.pdf`.

## Learning objectives

1. Express autoregressive generation as a policy acting over tokens.
2. Generate multiple completions per prompt and score them with verifiable rewards.
3. Construct group-relative advantages without a learned critic.
4. Implement the clipped policy-ratio objective and reference-policy KL penalty.
5. Track reward, exact-answer accuracy, formatting, entropy, KL, clipping, and zero-variance groups.
6. Explain what this controlled experiment does and does not reproduce from DeepSeek-R1.

## 1. Keep the paper's pipelines separate

### DeepSeek-R1-Zero

R1-Zero starts from DeepSeek-V3-Base and applies reasoning-oriented RL without preliminary SFT. Its automatically checked rewards incentivize correct answers and the requested output structure. The experiment shows that a strong base model can discover useful long-form reasoning behavior under outcome rewards, but it also develops readability and language-mixing problems.

### DeepSeek-R1

R1 is a multi-stage system: cold-start SFT, reasoning RL, rejection sampling plus a larger SFT stage, and another RL stage covering broader helpfulness and safety objectives. Therefore, R1's final performance cannot be attributed to GRPO alone.

This notebook most closely resembles a very small **R1-Zero-style algorithm experiment**: start from a randomly initialized policy, sample groups, apply rule-based rewards, and update with GRPO. A random tiny policy is not comparable to a pretrained LLM.

## 2. Translate language-model generation into RL

| RL term | Autoregressive LLM interpretation |
|---|---|
| Environment/context | Prompt $q$ and any verifier state |
| State $s_t$ | Prompt plus response prefix $o_{<t}$ |
| Action $a_t$ | Next response token $o_t$ |
| Policy $\pi_\theta$ | Decoder-only LM token distribution |
| Trajectory $o$ | Complete sampled response |
| Reward $r(q,o)$ | Verifier, format check, reward model, or combination |

The policy factorizes a response as

$$\pi_\theta(o\mid q)=\prod_{t=1}^{|o|}\pi_\theta(o_t\mid q,o_{<t}).$$

Unlike SFT, the response tokens are sampled from the policy. The reward evaluates the sampled trajectory; it does not provide a ground-truth token at every position. Policy gradients assign credit to sampled tokens that made high-reward trajectories more likely.

In [ ]:
from __future__ import annotations

import copy
import random

import torch
from torch import nn
from torch.distributions import Categorical
from torch.nn import functional as F

SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

print(f"PyTorch {torch.__version__}; device=cpu")

## 3. A verifiable toy environment

Each prompt contains two integers from 0 through 4. A completion contains exactly two actions:

```text
answer-token  <eos>
```

The verifier assigns `1.0` for the exact sum and `0.1` for the correct EOS format. There are no target response tokens in the RL update. The verifier only scores what the policy sampled.

This environment deliberately makes correctness objective and cheap. DeepSeek-R1 similarly concentrates reasoning RL on domains such as mathematics and code where answers can be checked, but its prompts, completions, base policy, and infrastructure are vastly more complex.

In [ ]:
MAX_INPUT = 4
MAX_ANSWER = 8
EOS = 9
INVALID = 10
NUM_ACTIONS = 11
START = 11  # input-only marker; the policy never emits it
RESPONSE_LENGTH = 2
ACTION_NAMES = [str(value) for value in range(MAX_ANSWER + 1)] + ["<eos>", "<invalid>"]

prompts = torch.tensor(
    [(left, right) for left in range(MAX_INPUT + 1) for right in range(MAX_INPUT + 1)],
    dtype=torch.long,
)

def score_completions(prompt_batch: torch.Tensor, actions: torch.Tensor):
    target = prompt_batch[:, 0] + prompt_batch[:, 1]
    correct = actions[:, 0].eq(target)
    correct_format = actions[:, 1].eq(EOS)
    rewards = correct.float() + 0.1 * correct_format.float()
    return rewards, correct, correct_format

def render_completion(actions: torch.Tensor) -> str:
    return " ".join(ACTION_NAMES[action] for action in actions.tolist())

print(f"Prompts: {len(prompts)}; action vocabulary: {ACTION_NAMES}")

## 4. Tiny autoregressive policy

The teaching policy uses a GRU cell rather than repeating the Transformer implementation from the first notebook. It is still autoregressive: the second action is conditioned on the prompt and first sampled action. GRPO only needs token probabilities and gradients, so the optimizer is unchanged when this policy is replaced by a decoder-only Transformer LLM.

During rollout, actions are sampled. During the GRPO update, the sampled actions are teacher-forced through the policy so their current log-probabilities can be recomputed.

In [ ]:
class TinyAutoregressivePolicy(nn.Module):
    def __init__(self, hidden_size: int = 48):
        super().__init__()
        self.number_embedding = nn.Embedding(MAX_INPUT + 1, hidden_size)
        self.action_embedding = nn.Embedding(NUM_ACTIONS + 1, hidden_size)
        self.initial_state = nn.Sequential(
            nn.Linear(2 * hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.recurrent_cell = nn.GRUCell(hidden_size, hidden_size)
        self.policy_head = nn.Linear(hidden_size, NUM_ACTIONS)

    def prompt_state(self, prompt_batch: torch.Tensor) -> torch.Tensor:
        left = self.number_embedding(prompt_batch[:, 0])
        right = self.number_embedding(prompt_batch[:, 1])
        return self.initial_state(torch.cat([left, right], dim=-1))

    def step(self, prompt_batch, previous_action, hidden_state=None):
        if hidden_state is None:
            hidden_state = self.prompt_state(prompt_batch)
        hidden_state = self.recurrent_cell(
            self.action_embedding(previous_action), hidden_state
        )
        return self.policy_head(hidden_state), hidden_state

    def forward(self, prompt_batch: torch.Tensor, actions: torch.Tensor):
        hidden_state = None
        previous_action = torch.full(
            (len(prompt_batch),), START, dtype=torch.long, device=prompt_batch.device
        )
        logits = []
        for position in range(RESPONSE_LENGTH):
            position_logits, hidden_state = self.step(
                prompt_batch, previous_action, hidden_state
            )
            logits.append(position_logits)
            previous_action = actions[:, position]
        return torch.stack(logits, dim=1)


def action_log_probs(policy, prompt_batch, actions):
    logits = policy(prompt_batch, actions)
    return F.log_softmax(logits, dim=-1).gather(
        dim=-1, index=actions.unsqueeze(-1)
    ).squeeze(-1)


@torch.no_grad()
def sample_groups(policy, unique_prompts, group_size, temperature=1.0):
    prompt_count = len(unique_prompts)
    expanded_prompts = (
        unique_prompts[:, None, :]
        .expand(prompt_count, group_size, 2)
        .reshape(-1, 2)
    )
    previous_action = torch.full((len(expanded_prompts),), START, dtype=torch.long)
    hidden_state = None
    actions, log_probs, entropies = [], [], []
    for _ in range(RESPONSE_LENGTH):
        logits, hidden_state = policy.step(expanded_prompts, previous_action, hidden_state)
        distribution = Categorical(logits=logits / temperature)
        previous_action = distribution.sample()
        actions.append(previous_action)
        log_probs.append(distribution.log_prob(previous_action))
        entropies.append(distribution.entropy())
    return (
        expanded_prompts,
        torch.stack(actions, dim=1),
        torch.stack(log_probs, dim=1),
        torch.stack(entropies, dim=1),
    )


@torch.no_grad()
def greedy_evaluate(policy, prompt_batch):
    previous_action = torch.full((len(prompt_batch),), START, dtype=torch.long)
    hidden_state = None
    actions = []
    for _ in range(RESPONSE_LENGTH):
        logits, hidden_state = policy.step(prompt_batch, previous_action, hidden_state)
        previous_action = logits.argmax(dim=-1)
        actions.append(previous_action)
    actions = torch.stack(actions, dim=1)
    rewards, correct, correct_format = score_completions(prompt_batch, actions)
    return {
        "reward": rewards.mean().item(),
        "accuracy": correct.float().mean().item(),
        "format": correct_format.float().mean().item(),
        "actions": actions,
    }


policy = TinyAutoregressivePolicy()
parameter_count = sum(parameter.numel() for parameter in policy.parameters())
print(f"Policy parameters: {parameter_count:,}")
print("Initial greedy metrics:", {k: v for k, v in greedy_evaluate(policy, prompts).items() if k != "actions"})

## 5. Group-relative advantages

For each prompt, sample $G$ completions and obtain rewards $r_1,\ldots,r_G$. GRPO uses the group as a prompt-specific baseline:

$$A_i = \frac{r_i-\mu_r}{\sigma_r+\epsilon_A}, \qquad \mu_r=\frac{1}{G}\sum_jr_j.$$

A completion is positive only relative to other samples for the same prompt. This avoids training a separate value critic, but introduces important behavior:

- If every completion has the same reward, every advantage is zero and that prompt supplies no policy-gradient signal.
- Very easy and very hard prompts can therefore become uninformative.
- Larger groups estimate relative outcomes more reliably but require more rollout generation.
- Reward scale within each group is normalized away; reward ordering remains.

In [ ]:
def group_relative_advantages(rewards: torch.Tensor, prompt_count: int, group_size: int):
    grouped = rewards.view(prompt_count, group_size)
    means = grouped.mean(dim=1, keepdim=True)
    standard_deviations = grouped.std(dim=1, keepdim=True, unbiased=False)
    normalized = (grouped - means) / (standard_deviations + 1e-4)
    normalized = torch.where(standard_deviations > 1e-6, normalized, 0.0)
    return normalized.reshape(-1), standard_deviations.squeeze(1)

demo_prompt = torch.tensor([[2, 2]])
demo_group_size = 8
demo_prompts, demo_actions, _, _ = sample_groups(policy, demo_prompt, demo_group_size)
demo_rewards, _, _ = score_completions(demo_prompts, demo_actions)
demo_advantages, demo_std = group_relative_advantages(
    demo_rewards, len(demo_prompt), demo_group_size
)
print("Samples for prompt 2 + 2:")
for actions, reward, advantage in zip(demo_actions, demo_rewards, demo_advantages):
    print(f"  {render_completion(actions):13s} reward={reward.item():.1f} advantage={advantage.item():+.3f}")
print(f"group reward standard deviation={demo_std.item():.3f}")

## 6. GRPO clipped objective and KL control

Rollouts are sampled by a fixed behavior policy $\pi_{\mathrm{old}}$. During optimization, recompute the sampled actions under the current policy and form a per-token importance ratio:

$$\rho_{i,t}(\theta)=\exp\left(\log\pi_\theta(o_{i,t}\mid q,o_{i,<t})-\log\pi_{\mathrm{old}}(o_{i,t}\mid q,o_{i,<t})\right).$$

The clipped surrogate is

$$\min\left(\rho_{i,t}A_i,\operatorname{clip}(\rho_{i,t},1-\epsilon,1+\epsilon)A_i\right).$$

Clipping discourages a sampled batch from moving the policy too far from the behavior policy. A separate frozen reference policy $\pi_{\mathrm{ref}}$ penalizes drift from the starting model. Following the sampled KL estimator used in this family of objectives, define

$$d_{i,t}=\log\pi_{\mathrm{ref}}(o_{i,t})-\log\pi_\theta(o_{i,t}),$$
$$\widehat D_{\mathrm{KL}}=\exp(d_{i,t})-d_{i,t}-1.$$

The implementation minimizes the negative mean of `clipped_surrogate - beta * sampled_kl`. One trajectory reward/advantage is broadcast across its sampled tokens; this is outcome-level rather than process-level credit assignment.

In [ ]:
def grpo_loss(
    policy,
    prompt_batch,
    actions,
    old_log_probs,
    reference_log_probs,
    advantages,
    clip_epsilon,
    kl_coefficient,
):
    current_log_probs = action_log_probs(policy, prompt_batch, actions)
    ratios = torch.exp(current_log_probs - old_log_probs)
    token_advantages = advantages[:, None]
    unclipped = ratios * token_advantages
    clipped = ratios.clamp(1 - clip_epsilon, 1 + clip_epsilon) * token_advantages
    clipped_surrogate = torch.minimum(unclipped, clipped)

    reference_minus_policy = reference_log_probs - current_log_probs
    sampled_kl = (
        torch.exp(reference_minus_policy) - reference_minus_policy - 1
    )
    objective = clipped_surrogate - kl_coefficient * sampled_kl
    loss = -objective.mean()
    diagnostics = {
        "loss": loss.item(),
        "kl": sampled_kl.mean().item(),
        "clip_fraction": (ratios.sub(1).abs() > clip_epsilon).float().mean().item(),
        "mean_ratio": ratios.mean().item(),
    }
    return loss, diagnostics


## 7. Complete on-policy training loop

Each update has two phases:

1. **Rollout:** freeze the current policy as the behavior policy for this batch, sample $G$ completions per prompt, store their old log-probabilities, and compute rewards and advantages.
2. **Optimization:** reuse those trajectories for several epochs. Recompute current log-probabilities, apply clipping and KL, backpropagate, clip gradients, and update the policy.

The reference policy is a frozen copy of the initial policy. We store old log-probabilities rather than a second old-policy model because the rollout batch is small and synchronous. Large systems must additionally manage generation/training workers, stale rollouts, sequence packing, distributed parameter updates, and reference inference.

In [ ]:
reference_policy = copy.deepcopy(policy).eval()
for parameter in reference_policy.parameters():
    parameter.requires_grad_(False)

optimizer = torch.optim.AdamW(policy.parameters(), lr=1.5e-2, weight_decay=1e-3)
GROUP_SIZE = 16
CLIP_EPSILON = 0.2
KL_COEFFICIENT = 0.02
OPTIMIZATION_EPOCHS = 4
UPDATES = 151
history = []

for update in range(UPDATES):
    # Rollout: these tensors define pi_old for the current batch.
    rollout_prompts, actions, old_log_probs, entropies = sample_groups(
        policy, prompts, GROUP_SIZE
    )
    rewards, correct, correct_format = score_completions(rollout_prompts, actions)
    advantages, reward_stds = group_relative_advantages(
        rewards, len(prompts), GROUP_SIZE
    )
    old_log_probs = old_log_probs.detach()
    with torch.no_grad():
        reference_log_probs = action_log_probs(
            reference_policy, rollout_prompts, actions
        )

    # Optimization: multiple gradient epochs over the fixed rollout batch.
    for _ in range(OPTIMIZATION_EPOCHS):
        loss, diagnostics = grpo_loss(
            policy,
            rollout_prompts,
            actions,
            old_log_probs,
            reference_log_probs,
            advantages,
            CLIP_EPSILON,
            KL_COEFFICIENT,
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm = nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

    greedy = greedy_evaluate(policy, prompts)
    record = {
        "update": update,
        "rollout_reward": rewards.mean().item(),
        "greedy_accuracy": greedy["accuracy"],
        "greedy_format": greedy["format"],
        "entropy": entropies.mean().item(),
        "zero_std_groups": (reward_stds < 1e-6).float().mean().item(),
        "gradient_norm": float(gradient_norm),
        **diagnostics,
    }
    history.append(record)
    if update % 10 == 0:
        print(
            f"u={update:3d} reward={record['rollout_reward']:.3f} "
            f"accuracy={record['greedy_accuracy']:.2f} format={record['greedy_format']:.2f} "
            f"KL={record['kl']:.3f} entropy={record['entropy']:.3f} "
            f"zero-groups={record['zero_std_groups']:.2f}"
        )

final_metrics = greedy_evaluate(policy, prompts)
assert final_metrics["accuracy"] == 1.0
assert final_metrics["format"] == 1.0
print("Final greedy accuracy and format checks: 100%")

In [ ]:
final_actions = final_metrics["actions"]
for prompt, actions in zip(prompts, final_actions):
    left, right = prompt.tolist()
    print(f"{left} + {right} -> {render_completion(actions)}")

## 8. Map the notebook to DeepSeek-R1

| Notebook | DeepSeek-R1 analogue | Important difference |
|---|---|---|
| Two-integer prompt | Math, code, and STEM prompt | Real prompts require long semantic reasoning |
| Two-token GRU policy | DeepSeek-V3-Base / R1 policy | Real policy is a very large decoder-only MoE Transformer |
| 16 samples per prompt | Grouped response rollouts | Real completions may contain thousands of tokens |
| Exact sum reward | Rule-based accuracy verifier | Real verifiers cover answers, programs, and task-specific checks |
| EOS reward | Structural format reward | R1 uses explicit reasoning/final-answer structure and other pipeline controls |
| Group reward normalization | GRPO group-relative baseline | Same concept, radically different scale and sampling distribution |
| Frozen initial policy | Reference model | Real reference inference is a major systems cost |
| Stored rollout log-probs | Old behavior policy | Distributed rollouts introduce policy staleness |
| Greedy exact accuracy | pass@1-style evaluation | Reasoning evaluation also studies sampling, robustness, style, and contamination |

The notebook demonstrates that the estimator can optimize a sparse sequence reward. It does not show emergent reasoning, self-reflection, inference-time scaling, or generalization. The policy memorizes a finite table of arithmetic prompts.

## 9. Compute and storage implications

GRPO removes the learned critic/value model used in conventional PPO, but RL training remains substantially more expensive than SFT.

### Models and stored tensors

- **Trainable policy:** weights, gradients, optimizer states, and training activations.
- **Reference policy:** frozen weights plus forward activations needed only transiently.
- **Old policy:** may be a rollout snapshot; this notebook stores old token log-probabilities instead.
- **Rollout batch:** prompt/response tokens, masks, old log-probabilities, reference log-probabilities, rewards, and advantages.
- **Generation KV cache:** scales with concurrent sequences and generated context length.

For $B$ unique prompts, group size $G$, and response length $R$, rollout generation produces approximately $BGR$ response-token decisions. Autoregressive generation is sequential in $R$. Reusing the rollout batch for $K$ optimization epochs adds roughly $K$ policy forward/backward passes over the prompt-response sequences, plus reference-policy scoring.

Removing a critic saves critic parameters, optimizer state, inference, and value activations. It does **not** remove grouped sampling, reference scoring, policy training state, or rollout storage. At LLM scale, rollout throughput, long and variable response lengths, distributed synchronization, and stale-policy control are often as important as the loss equation.

## 10. Critical questions and exercises

1. Set `GROUP_SIZE = 2`, `4`, `8`, and `32`. Compare reward variance, zero-variance groups, and updates needed for convergence.
2. Remove the format reward. Does the policy still learn EOS, and what does that say about underspecified rewards?
3. Remove the KL term, then increase it substantially. Compare exploration, convergence, and policy drift.
4. Set `OPTIMIZATION_EPOCHS` high enough to create a large clip fraction. Explain why stale rollouts eventually become unsafe to reuse.
5. Corrupt the verifier for a subset of prompts. Observe reward hacking: the policy optimizes the implemented reward, not the intended task.
6. Add prompts whose answers are outside the action vocabulary. Show why impossible groups yield no useful relative signal.
7. Create a cold-start policy with SFT on a small subset, then run the same RL loop. Compare format stability and sample efficiency with random-start RL.
8. Replace sequence-level advantages with token-specific process rewards. Identify the new supervision and failure modes required.
9. Replace the GRU with the causal Transformer from notebook 01 without changing the GRPO code.
10. Re-read the paper's unsuccessful attempts and classify each issue as reward design, search/optimization, model capacity, or infrastructure.